In [5]:
import os
import pickle
import torch

# Example usage
folder_path = "/home/laios/BERT_english_activations"


In [12]:
import sys,os
sys.path.append('../')
sys.path.append('../../')
from tqdm import tqdm
from utils import list_folder
from collections import defaultdict

import os
import pickle
import torch
from collections import defaultdict
from tqdm import tqdm

def collect_bert_data(
    input_path,
    n_loaded_tokens,
    n_files,
    mask=0,
    take_last=True
):
    def take_last_hidden_states(hidden_states, mask, n_loaded_tokens):
        # hidden_states shape: [layers, batch, seq_len, dim]
        return hidden_states[:, :, -n_loaded_tokens-mask:-mask]

    def take_first_hidden_states(hidden_states, mask, n_loaded_tokens):
        return hidden_states[:, :, mask:n_loaded_tokens+mask]

    files = sorted([f for f in os.listdir(input_path) if f.endswith(".pkl")])[:n_files]
    all_hidden_states = defaultdict(list)

    slice_hidden_states = take_last_hidden_states if take_last else take_first_hidden_states

    for fname in tqdm(files, desc="Collect File"):
        with open(os.path.join(input_path, fname), "rb") as f:
            hidden_states = pickle.load(f)  # [layers, batch, seq_len, dim]

        # Ensure all sequences are long enough
        assert hidden_states.shape[2] >= n_loaded_tokens + mask, \
            f"{fname} has sequence length {hidden_states.shape[2]} < {n_loaded_tokens + mask}"

        sliced = slice_hidden_states(hidden_states, mask, n_loaded_tokens)  # [layers, batch, tokens, dim]

        for layer_idx in range(sliced.shape[0]):
            layer_tensor = sliced[layer_idx]  # [batch, tokens, dim]
            all_hidden_states[f"layer_{layer_idx}"].append(layer_tensor)

    # Concatenate tensors for each layer
    for layer, tensors in all_hidden_states.items():
        all_hidden_states[layer] = torch.cat(tensors, dim=0)  # [total_batch, tokens, dim]

    return all_hidden_states



In [18]:
data = collect_bert_data(folder_path,38,30,2)

Collect File: 100%|██████████| 30/30 [02:01<00:00,  4.05s/it]


In [16]:
data['layer_21'].shape

torch.Size([300, 20, 1024])